In [1]:
import ee
import geemap
import geopandas as gpd
import pandas as pd
import requests
import requests, tempfile, ee, geemap, geopandas as gpd
import pandas as pd
import ipywidgets as widgets
import io
from contextlib import redirect_stdout
from ipyleaflet import link
from shapely.geometry import shape
from IPython.display import display

# 1. Inicializar Earth Engine
try:
    ee.Initialize()
    print("✅ Earth Engine conectado con éxito.")
except Exception as e:
    print(f"❌ Error al conectar EE: {e}")

# 2. Mostrar el mapa interactivo
print("Cargando mapa...")
Map = geemap.Map(center=[33.58, -101.83], zoom=14)
Map

# Habilitar interactividad de mapas en Colab
try:
    ee.Initialize(project='my-project-12126-484118')
    print("Earth Engine inicializado.")
except Exception as e:
    print("Autenticando Earth Engine...")
    ee.Authenticate()
    ee.Initialize(project='my-project-12126-484118')

print("\n--- PASO 1: DIBUJA TU LOTE ---")
print("Usa el polígono de la izquierda para dibujar. Cuando termines, pasa a la Celda 2.")
Draw_Map = geemap.Map(center=[33.584, -101.845], zoom=14)
Draw_Map.add_basemap('SATELLITE')
display(Draw_Map)

if Draw_Map.user_roi:
    print("Coordinates of your drawn polygon:")
    # The user_roi is an Earth Engine object, so .getInfo() is needed to get client-side representation
    print(Draw_Map.user_roi.getInfo()['coordinates'])
else:
    print("No polygon detected. Please draw a polygon on the map in the first cell and run this cell again.")

✅ Earth Engine conectado con éxito.
Cargando mapa...
Earth Engine inicializado.

--- PASO 1: DIBUJA TU LOTE ---
Usa el polígono de la izquierda para dibujar. Cuando termines, pasa a la Celda 2.


Map(center=[33.584, -101.845], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topri…

No polygon detected. Please draw a polygon on the map in the first cell and run this cell again.


In [2]:
# ========================================
# 2. SOIL SPATIAL DATA EXTRACTION (WFS)
# ========================================
# Wait for user input from the map
if Draw_Map.user_roi is None:
    raise ValueError("❌ No polygon detected. Please draw a polygon on the map above and rerun this cell.")

print("\n--- STEP 2: DOWNLOADING SPATIAL SOIL DATA (USDA-NRCS) ---")

# 1. Extract coordinates from the drawn polygon
roi_coords = Draw_Map.user_roi.getInfo()['coordinates'][0]
lote_geom = ee.Geometry.Polygon([roi_coords])
centro_lote = lote_geom.centroid().coordinates().getInfo()[::-1] # [lat, lon]

# Convert EE coordinates to a Shapely Polygon
from shapely.geometry import Polygon
shapely_poly = Polygon(roi_coords)

# Calculate Bounding Box for the WFS request (lon_min, lat_min, lon_max, lat_max)
minx, miny, maxx, maxy = shapely_poly.bounds
bbox_str = f"{minx},{miny},{maxx},{maxy}" # Corrected BBOX order: lon_min,lat_min,lon_max,lat_max

# 2. Query the USDA Web Feature Service (WFS)
wfs_url = "https://sdmdataaccess.nrcs.usda.gov/Spatial/SDMNAD83Geographic.wfs"
params = {
    'SERVICE': 'WFS',
    'VERSION': '1.0.0',
    'REQUEST': 'GetFeature',
    'TYPENAME': 'MapunitPoly',
    'BBOX': bbox_str,
    'SRSNAME': 'EPSG:4326',
    'OUTPUTFORMAT': 'GML3' # Changed from 'application/json' to 'GML3'
}

print("📡 Connecting to USDA Soil Data Mart (WFS)...")
import io
response = requests.get(wfs_url, params=params, verify=False, timeout=30)

if response.status_code != 200:
    raise ConnectionError(f"❌ Failed to retrieve data from USDA: {response.text}")

# 3. Process the GML into a GeoDataFrame and clip it to the exact field boundary
gdf_soils = gpd.read_file(io.BytesIO(response.content))

if gdf_soils.empty:
    raise ValueError("⚠️ No soil data found in this specific area.")

gdf_lote = gpd.GeoDataFrame(index=[0], crs='epsg:4326', geometry=[shapely_poly])

# Intersect (clip) the soil map units with the drawn field polygon
gdf_clipped = gpd.overlay(gdf_soils, gdf_lote, how='intersection')
print("✅ Soil spatial data successfully retrieved and clipped to field boundary.")


# 5. PREPARACIÓN DE MAPAS
centroid = lote_geom.centroid(maxError=1).coordinates().getInfo()
centro_lote = [centroid[1], centroid[0]]

# Preparamos los datos de suelo para el mapa (capa visual amarilla)
gdf_4p = gdf_clipped[['mukey', 'musym', 'geometry']].copy()

def crear_mapa(capa, nombre):
    m = geemap.Map(center=centro_lote, zoom=14, add_google_map=False)
    m.add_basemap('SATELLITE')
    if capa: m.addLayer(capa, ndvi_vis, nombre)

    # Silenciamos el output ruidoso de geemap
    with redirect_stdout(io.StringIO()):
        m.add_gdf(gdf_4p, layer_name="Suelos",
                  style={'color': 'yellow', 'weight': 1, 'fillOpacity': 0},
                  hover_style={'fillOpacity': 0.3})
    return m


# Esto crea el mapa (sin capa de NDVI por ahora) y lo guarda en la variable 'mi_mapa'
mi_mapa = crear_mapa(None, "Mapa Base")

# Esto finalmente lo muestra en la pantalla
display(mi_mapa)


--- STEP 2: DOWNLOADING SPATIAL SOIL DATA (USDA-NRCS) ---
📡 Connecting to USDA Soil Data Mart (WFS)...
✅ Soil spatial data successfully retrieved and clipped to field boundary.


Map(center=[33.58269842580678, -101.81219100000774], controls=(WidgetControl(options=['position', 'transparent…

In [3]:
# ========================================
# 3. SOIL REPORT GENERATION (SDA TABULAR DATA + UC DAVIS LINKS)
# ========================================
print("\n--- STEP 3: GENERATING AGRONOMIC SOIL REPORT ---")

# 1. Calculate Area (Hectares) for each Map Unit (MUKEY)
# We reproject to EPSG:3857 (Web Mercator) just for accurate area calculation in meters
gdf_temp = gdf_clipped.copy()
gdf_temp['hectareas'] = gdf_temp.to_crs(epsg=3857).area / 10000

# Group by MUKEY and MUSYM (Map Unit Symbol) to get total area per soil type
resumen_mukey = gdf_temp.groupby(['mukey', 'musym']).agg({'hectareas': 'sum'}).reset_index()
resumen_mukey = resumen_mukey.sort_values(by='hectareas', ascending=False)
total_ha = resumen_mukey['hectareas'].sum()

# Format MUKEYs for the SQL query
muke_list = resumen_mukey['mukey'].unique().tolist()
mukeys_str = str(tuple(muke_list)).replace(",)", ")")

# 2. Build the SQL Query for the Soil Data Access (SDA) service
# This extracts surface horizon data (0-20cm approx): Sand, Silt, Clay, and Organic Matter
sql_query = f"""
SELECT c.mukey, c.compname, c.comppct_r, ch.sandtotal_r, ch.silttotal_r, ch.claytotal_r, ch.om_r
FROM component AS c
LEFT JOIN chorizon AS ch ON c.cokey = ch.cokey
WHERE c.mukey IN {mukeys_str}
AND ch.hzdept_r = (SELECT MIN(hzdept_r) FROM chorizon WHERE chorizon.cokey = c.cokey)
"""

# 3. Execute the POST request to the SDA Tabular endpoint
series_data = pd.DataFrame()
try:
    print("📡 Fetching textural and organic matter properties (SDA)...")
    url_sda = "https://sdmdataaccess.nrcs.usda.gov/tabular/post.rest"
    payload = {"query": sql_query, "format": "JSON"}
    r = requests.post(url_sda, data=payload, verify=False, timeout=30)

    if r.status_code == 200:
        json_data = r.json()
        if 'Table' in json_data:
            # Map the columns matching the SELECT statement
            series_data = pd.DataFrame(json_data['Table'], columns=['mukey', 'serie', 'pct', 'arena', 'limo', 'arcilla', 'mo'])
    else:
        print(f"⚠️ USDA Server returned status code: {r.status_code}")
except Exception as e:
    print(f"❌ Error connecting to USDA Tabular DB: {e}")

# 4. Construct the Interactive HTML Report
from IPython.display import HTML
import urllib

html_output = f"<div style='font-family: sans-serif; max-width: 950px;'><h2 style='color: #2c3e50; border-bottom: 3px solid #2c3e50;'>📊 Soil Report: Composition and Texture</h2>"

for _, fila in resumen_mukey.iterrows():
    # Filter tabular data for the current Map Unit
    detalles = series_data[series_data['mukey'].astype(str) == str(fila['mukey'])] if not series_data.empty else pd.DataFrame()
    pct_lote = (fila['hectareas'] / total_ha) * 100

    html_output += f"""
    <details style=\"margin-bottom: 12px; border: 1px solid #bdc3c7; border-radius: 6px; overflow: hidden;\">
        <summary style=\"background-color: #2c3e50; padding: 15px; font-weight: bold; cursor: pointer; color: #fff;\">
            📍 Unit: {fila['musym']} — {fila['hectareas']:.2f} ha ({pct_lote:.1f}%)
        </summary>
        <div style=\"padding: 15px; background: #fff;\">
            <table style=\"width: 100%; border-collapse: collapse;\">
                <thead>
                    <tr style=\"background-color: #ecf0f1; color: #2c3e50;\">
                        <th style=\"padding: 10px; text-align: left; border: 1px solid #ddd;\">Soil Series</th>
                        <th style=\"border: 1px solid #ddd;\">%</th>
                        <th style=\"border: 1px solid #ddd;\">Sand %</th>
                        <th style=\"border: 1px solid #ddd;\">Silt %</th>
                        <th style=\"border: 1px solid #ddd;\">Clay %</th>
                        <th style=\"border: 1px solid #ddd; color:#e67e22;\">O.M. %</th>
                    </tr>
                </thead>
                <tbody>
    """
    for _, s in detalles.iterrows():
        nombre_s = str(s['serie']).strip()
        # URL encoding for UC Davis SoilWeb links (e.g., 'Acuff' -> 'acuff')
        nombre_s_url = urllib.parse.quote(nombre_s.lower())
        url_ficha = f"https://casoilresource.lawr.ucdavis.edu/sde/?series={nombre_s_url}#osd"

        html_output += f"""
                <tr style=\"border-bottom: 1px solid #eee;\">
                    <td style=\"padding: 10px; font-weight: bold; border: 1px solid #ddd;\">
                        <a href=\"{url_ficha}\" target=\"_blank\" style=\"color: #3498db; text-decoration: none;\">{nombre_s}</a> 🔗
                    </td>
                    <td style=\"text-align: center; border: 1px solid #ddd; color: #333; \">{s['pct']}%</td>
                    <td style=\"text-align: center; border: 1px solid #ddd; color: #333; \">{s['arena'] if s['arena'] else '-' }</td>
                    <td style=\"text-align: center; border: 1px solid #ddd; color: #333; \">{s['limo'] if s['limo'] else '-' }</td>
                    <td style=\"text-align: center; border: 1px solid #ddd; color: #333; \">{s['arcilla'] if s['arcilla'] else '-' }</td>
                    <td style=\"text-align: center; border: 1px solid #ddd; color: #d35400; font-weight: bold;\">{s['mo'] if s['mo'] else '-' }</td>
                </tr>"""
    html_output += "</tbody></table></div></details>"

display(HTML(html_output + "</div>"))


--- STEP 3: GENERATING AGRONOMIC SOIL REPORT ---
📡 Fetching textural and organic matter properties (SDA)...


Soil Series,%,Sand %,Silt %,Clay %,O.M. %
Amarillo 🔗,50%,67.5,16.4,16.1,0.9
Urban land 🔗,35%,-,-,-,-
Ustorthents 🔗,5%,36,34,30,0.3
Acuff 🔗,10%,50.9,30,19.1,2.5
Soil Series,%,Sand %,Silt %,Clay %,O.M. %
Amarillo 🔗,10%,67.5,16.4,16.1,0.9
Ustorthents 🔗,5%,36,34,30,0.3
Urban land 🔗,35%,-,-,-,-
Acuff 🔗,50%,50.9,30,19.1,2.5
Soil Series,%,Sand %,Silt %,Clay %,O.M. %
